# The 1D Cluster Model as a Symmetry-Protected Topological Phase

$$H \;=\; -\sum_j Z_{j-1}X_j Z_{j+1}$$

protected by $\mathbb{Z}_2\times\mathbb{Z}_2$ generated by
$$U_{\mathrm{odd}}=\prod_{j\ \mathrm{odd}}X_j,\qquad U_{\mathrm{even}}=\prod_{j\ \mathrm{even}}X_j .$$

This notebook makes five things visible.

| | what | figure |
|---|---|---|
| 1 | the model and its stabilizers | Fig. 1 |
| 2 | open chain ($N$ odd and even) has a **4-fold** ground space; the ring has a **unique** one | Fig. 2, 3 |
| 3 | the degeneracy survives symmetric perturbations and dies under symmetry-breaking ones | Fig. 4 |
| 4 | the entanglement spectrum carries the same information | Fig. 5 |
| 5 | the symmetry acts **projectively** on each edge | Fig. 6 |

Everything is exact diagonalization; no approximation anywhere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.axisbelow": True,
})
CB = {"blue":"#2f6fb5","red":"#c0392b","green":"#27865b","orange":"#d4801f",
      "purple":"#7d5ba6","grey":"#7a7a7a"}

I2 = np.eye(2); X = np.array([[0.,1],[1,0]]); Z = np.diag([1.,-1])

def op(N, d):
    M = np.array([[1.]])
    for j in range(N):
        M = np.kron(M, d.get(j, I2))
    return M

def stabilizers(N, ring=False):
    if ring:
        return [op(N, {(j-1) % N: Z, j: X, (j+1) % N: Z}) for j in range(N)]
    return [op(N, {j-1: Z, j: X, j+1: Z}) for j in range(1, N-1)]

def H_cluster(N, ring=False):
    return -sum(stabilizers(N, ring))

def sum_single(N, P):
    return sum(op(N, {j: P}) for j in range(N))

def U_sub(N, parity):
    return op(N, {j: X for j in range(parity, N, 2)})

def degeneracy(w, tol=1e-8):
    return int(np.sum(np.isclose(w, w[0], atol=tol)))

def entanglement_spectrum(H, N, k=8):
    w, v = np.linalg.eigh(H)
    psi = v[:, 0].reshape(2**(N//2), 2**(N//2))
    sv = np.linalg.svd(psi, compute_uv=False)**2
    return -np.log(sv[sv > 1e-14][:k])

print("helpers ready")

## Sanity checks

Three facts underpin everything. Each is one line of code.

1. the stabilizers $S_j=Z_{j-1}X_jZ_{j+1}$ mutually commute, so the model is exactly solvable;
2. the symmetry generators commute with every $S_j$;
3. on a **ring**, $\prod_{j\ \mathrm{odd}}S_j = U_{\mathrm{odd}}$ — the symmetry is a product of stabilizers, hence acts trivially on the ground state. On an **open chain** this identity fails by boundary terms, and that failure is the whole story.

In [ ]:
N = 8
S = stabilizers(N)
print("stabilizers commute pairwise      :", all(np.allclose(a@b, b@a) for a in S for b in S))
print("symmetry commutes with stabilizers:",
      all(np.allclose(U@s, s@U) for U in (U_sub(N,0), U_sub(N,1)) for s in S))

Sr = stabilizers(N, ring=True)
prod_odd = np.eye(2**N)
for j in range(0, N, 2):
    prod_odd = prod_odd @ Sr[j]
print("ring:  prod_{j odd} S_j == U_odd   :", np.allclose(prod_odd, U_sub(N,0)))

# open chain: the same product leaves uncancelled Z's at the two ends
prod_open = np.eye(2**N)
for j in range(2, N-1, 2):
    prod_open = prod_open @ S[j-1]
resid = prod_open @ U_sub(N,0)
print("open:  U_odd = (X1 Z2)·(Z_N) on GS : see Fig. 6")

## Fig. 1 — the model

The chain, one stabilizer, the two sublattice symmetries, and the edge operators that will turn out to carry the degeneracy.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 5.4))
Nd = 9
xs = np.arange(Nd)
for ax in axes:
    ax.set_xlim(-5.4, 9.4); ax.set_ylim(-1.15, 1.15)
    ax.set_aspect("equal"); ax.axis("off")

def draw_chain(ax, labels=True):
    ax.plot(xs, np.zeros(Nd), "-", color=CB["grey"], lw=1.2, zorder=1)
    for j in xs:
        ax.add_patch(Circle((j, 0), 0.22,
                     color=CB["blue"] if j % 2 == 0 else CB["orange"], zorder=3))
        if labels:
            ax.text(j, -0.95, f"{j+1}", ha="center", va="center", fontsize=8,
                    color=CB["grey"])

# (1) stabilizer
ax = axes[0]; draw_chain(ax)
for j, lab, col in [(3,"Z",CB["green"]),(4,"X",CB["red"]),(5,"Z",CB["green"])]:
    ax.text(j, 0.55, lab, ha="center", fontsize=13, fontweight="bold", color=col)
ax.add_patch(FancyBboxPatch((2.62,-0.4),2.76,1.28, boxstyle="round,pad=0.05",
        fc="none", ec=CB["red"], lw=1.3, ls="--", zorder=2))
ax.text(-5.2, 0.25, r"$S_j=Z_{j-1}X_jZ_{j+1}$", fontsize=12, color=CB["red"], va="center")

# (2) the two sublattice symmetries
ax = axes[1]; draw_chain(ax, labels=False)
for j in xs[::2]:
    ax.text(j, 0.55, "X", ha="center", fontsize=12, fontweight="bold", color=CB["blue"])
for j in xs[1::2]:
    ax.text(j, -0.72, "X", ha="center", fontsize=12, fontweight="bold", color=CB["orange"])
ax.text(-5.2, 0.55, r"$U_{\mathrm{odd}}=\prod_{j\,\mathrm{odd}}X_j$",
        fontsize=11.5, color=CB["blue"], va="center")
ax.text(-5.2,-0.72, r"$U_{\mathrm{even}}=\prod_{j\,\mathrm{even}}X_j$",
        fontsize=11.5, color=CB["orange"], va="center")

# (3) edge operators
ax = axes[2]; draw_chain(ax, labels=False)
for (j, lab, y, col) in [(0,"Z",0.55,CB["purple"]), (8,"Z",0.55,CB["purple"]),
                         (0,"X",-0.72,CB["green"]), (1,"Z",-0.72,CB["green"]),
                         (7,"Z",-0.72,CB["green"]), (8,"X",-0.72,CB["green"])]:
    ax.text(j, y, lab, ha="center", fontsize=12, fontweight="bold", color=col)
ax.text(-5.2, 0.55, r"$\sigma^z_L=Z_1$", fontsize=11.5, color=CB["purple"], va="center")
ax.text(-5.2,-0.72, r"$\sigma^x_L=X_1Z_2$", fontsize=11.5, color=CB["green"], va="center")
ax.text(-5.2,-1.05, "(mirrored on the right)", fontsize=9, color=CB["grey"], va="center")

axes[0].set_title("Fig. 1   Stabilizer, symmetry, edge operators", fontsize=12.5,
                  loc="left", pad=4)
plt.tight_layout(); plt.show()

## Fig. 2 — spectra

An open chain of $N$ sites admits stabilizers $S_j$ only for $j=2,\dots,N-1$, that is $N-2$ of them, so

$$\dim(\text{ground space}) = \frac{2^N}{2^{N-2}} = 4 .$$

This holds for **both parities of $N$**. On a ring all $N$ stabilizers exist and the ground state is unique. The contrast is the point: the degeneracy is an *edge* phenomenon, not a bulk one.

In [ ]:
cases = [(7, False, "(a) open chain, $N=7$ (odd)"),
         (8, False, "(b) open chain, $N=8$ (even)"),
         (8, True,  "(c) ring, $N=8$")]

def levels(w, tol=1e-7):
    # group eigenvalues into (energy, multiplicity)
    out = []
    for e in w:
        if out and abs(e - out[-1][0]) < tol: out[-1][1] += 1
        else: out.append([e, 1])
    return out

fig, axes = plt.subplots(1, 3, figsize=(11, 4.4))
for ax, (N, ring, title) in zip(axes, cases):
    w = np.linalg.eigvalsh(H_cluster(N, ring)); w = w - w[0]
    lv = [(e, m) for e, m in levels(w) if e < 4.6]
    d = lv[0][1]
    for e, m in lv:
        col = CB["red"] if abs(e) < 1e-7 else CB["grey"]
        lw  = 3.0 if abs(e) < 1e-7 else 1.4
        xs_ = np.linspace(0.16, 0.74, m) if m > 1 else np.array([0.45])
        hw  = min(0.030, 0.58/(2.2*m)) if m > 1 else 0.030
        for x in xs_:
            ax.hlines(e, x-hw, x+hw, color=col, lw=lw)
        ax.text(0.955, e, f"$\\times${m}", fontsize=8.5, va="center", ha="right",
                color=col, alpha=0.95)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_xlim(0.06, 1.0)
    ax.set_ylabel(r"$E-E_0$"); ax.set_ylim(-0.5, 4.7)
    ax.text(0.5, -0.38, f"ground space: {d}-fold", ha="center", fontsize=11,
            color=CB["red"], fontweight="bold")
fig.suptitle("Fig. 2   Each tick is one state — the ground space is 4-fold on a chain, 1-fold on a ring",
             fontsize=12.5, y=1.02)
plt.tight_layout(); plt.show()

## Fig. 3 — degeneracy versus system size

The ground-space dimension is $4$ for every open chain and $1$ for every ring, independently of $N$ and of its parity. A degeneracy that does not grow with $N$ and disappears when the boundary is removed is the signature of edge modes.

In [ ]:
Ns = [5,6,7,8,9,10]
deg_open, deg_ring, gap_open = [], [], []
for N in Ns:
    w = np.linalg.eigvalsh(H_cluster(N)); d = degeneracy(w)
    deg_open.append(d); gap_open.append(w[d]-w[0])
    deg_ring.append(degeneracy(np.linalg.eigvalsh(H_cluster(N, ring=True))))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
wdt = 0.36
ax1.bar(np.array(Ns)-wdt/2, deg_open, wdt, label="open chain", color=CB["red"])
ax1.bar(np.array(Ns)+wdt/2, deg_ring, wdt, label="ring", color=CB["blue"])
ax1.set_xlabel("$N$"); ax1.set_ylabel("ground-space dimension")
ax1.set_yticks([0,1,2,3,4]); ax1.legend(frameon=False)
ax1.set_title("degeneracy is 4 (open) / 1 (ring), for all $N$", fontsize=11)
for N, d in zip(Ns, deg_open):
    ax1.text(N-wdt/2, d+0.08, str(d), ha="center", fontsize=9, color=CB["red"])

ax2.plot(Ns, gap_open, "o-", color=CB["green"], lw=1.6)
ax2.set_xlabel("$N$"); ax2.set_ylabel("bulk gap"); ax2.set_ylim(0, 2.6)
ax2.set_title("the bulk gap stays open", fontsize=11)
fig.suptitle("Fig. 3   The degeneracy does not scale with volume", fontsize=12.5, y=1.04)
plt.tight_layout(); plt.show()

## Fig. 4 — what "protected" means

Add a field and watch the four ground levels.

$$H(\lambda)=-\sum_j Z_{j-1}X_jZ_{j+1}\;-\;\lambda\sum_j P_j ,\qquad P=X\ \text{or}\ Z .$$

* $P=X$ **preserves** $\mathbb{Z}_2\times\mathbb{Z}_2$, since $X_j$ commutes with both $\prod X$.
* $P=Z$ **breaks** it, since $Z_j$ anticommutes with the $X$ on its own sublattice.

The prediction is that the splitting of the four ground levels is exponentially small in $N$ in the first case and linear in $\lambda$ in the second.

In [ ]:
N = 10
H0 = H_cluster(N)
PX, PZ = sum_single(N, X), sum_single(N, Z)
lams = np.linspace(0, 0.6, 41)

def sweep(P, nlev=8):
    out = []
    for lam in lams:
        out.append(np.linalg.eigvalsh(H0 - lam*P)[:nlev])
    return np.array(out)

specX, specZ = sweep(PX), sweep(PZ)

fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.4),
                         gridspec_kw={"height_ratios":[1.35,1]})
for k, (spec, ttl, col) in enumerate(
        [(specX, r"symmetric:  $-\lambda\sum_j X_j$", CB["green"]),
         (specZ, r"symmetry-breaking:  $-\lambda\sum_j Z_j$", CB["red"])]):
    ax = axes[0, k]
    ref = spec[:, 0]
    for i in range(4):
        ax.plot(lams, spec[:, i]-ref, lw=2.2, color=col, alpha=0.9,
                label="ground quartet" if i == 0 else None)
    for i in range(4, 8):
        ax.plot(lams, spec[:, i]-ref, lw=1.0, color=CB["grey"], alpha=0.6,
                label="excited states" if i == 4 else None)
    ax.set_title(ttl, fontsize=11.5, color=col)
    ax.set_xlabel(r"$\lambda$"); ax.set_ylabel(r"$E-E_0$")
    ax.set_ylim(-0.15, 2.8); ax.legend(frameon=False, fontsize=9, loc="upper left")

    ax = axes[1, k]
    split = spec[:, 3]-spec[:, 0]
    ax.semilogy(lams, np.maximum(split, 1e-16), lw=2.2, color=col)
    ax.set_xlabel(r"$\lambda$"); ax.set_ylabel(r"$E_3-E_0$   (log)")
    ax.set_ylim(1e-9, 5)
    ax.axhline(1e-8, color=CB["grey"], ls=":", lw=1)
    verdict = "degeneracy survives" if k == 0 else "degeneracy destroyed at once"
    ax.text(0.03, 0.9, verdict, transform=ax.transAxes, fontsize=10.5,
            color=col, fontweight="bold", va="top")
fig.suptitle("Fig. 4   The four-fold degeneracy is protected by the symmetry, not by the gap",
             fontsize=12.5, y=1.02)
plt.tight_layout(); plt.show()

print(f"splitting at lambda=0.05 :  symmetric {specX[4,3]-specX[4,0]:.2e}"
      f"   |   breaking {specZ[4,3]-specZ[4,0]:.2e}")

### Is it really *exponentially* small?

A splitting that shrinks with system size is a finite-size artefact; one that does not is a genuine lifting of the degeneracy. Fixing $\lambda$ and increasing $N$ separates the two.

In [ ]:
Ns_fs = [6, 8, 10, 12]
lam_fs = 0.3
splitX, splitZ = [], []
for N in Ns_fs:
    H0n = H_cluster(N)
    wX = np.linalg.eigvalsh(H0n - lam_fs*sum_single(N, X))[:4]
    wZ = np.linalg.eigvalsh(H0n - lam_fs*sum_single(N, Z))[:4]
    splitX.append(wX[3]-wX[0]); splitZ.append(wZ[3]-wZ[0])

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.semilogy(Ns_fs, splitX, "o-", lw=2, ms=7, color=CB["green"],
            label=r"symmetric  $-\lambda\sum X_j$")
ax.semilogy(Ns_fs, splitZ, "s-", lw=2, ms=7, color=CB["red"],
            label=r"breaking  $-\lambda\sum Z_j$")
ax.set_xlabel("$N$"); ax.set_ylabel(r"$E_3-E_0$   at $\lambda=0.3$")
ax.set_xticks(Ns_fs); ax.legend(frameon=False, fontsize=10)
ax.set_title("Fig. 4b   The protected splitting decays with $N$; the other does not",
             fontsize=11.5, loc="left")
plt.tight_layout(); plt.show()

for N, a, b in zip(Ns_fs, splitX, splitZ):
    print(f"N={N:2d}   symmetric {a:.3e}    breaking {b:.3e}")

## Fig. 5 — entanglement spectrum

Cut the ring in half and record $-\ln\lambda_i$ for the Schmidt weights. At the fixed point the spectrum is exactly flat with four equal levels — one factor of two for each of the two cuts, and each factor of two is the projective representation carried by that cut.

Switching on a perturbation, the spectrum spreads. The rate at which it spreads distinguishes the two cases.

In [ ]:
N = 10
H0r = H_cluster(N, ring=True)
PXr, PZr = sum_single(N, X), sum_single(N, Z)
lam_es = np.linspace(0, 0.6, 25)

def es_sweep(P, k=4):
    return np.array([entanglement_spectrum(H0r - lam*P, N, k) for lam in lam_es])

esX, esZ = es_sweep(PXr), es_sweep(PZr)

fig = plt.figure(figsize=(11, 3.9))
gs = fig.add_gridspec(1, 3, width_ratios=[0.8, 1.1, 1.1])

ax = fig.add_subplot(gs[0])
e0 = entanglement_spectrum(H0r, N, 6)
for e in e0:
    ax.hlines(e, 0.1, 0.9, color=CB["purple"], lw=3)
ax.text(0.5, e0[0]+0.09, f"{len(e0)}-fold", ha="center", color=CB["purple"],
        fontsize=11, fontweight="bold")
ax.set_xticks([]); ax.set_ylabel(r"$-\ln\lambda_i$")
ax.set_ylim(e0[0]-0.55, e0[0]+0.55); ax.set_title(r"$\lambda=0$: flat", fontsize=11)

for k, (es, ttl, col) in enumerate(
        [(esX, r"symmetric  $-\lambda\sum X_j$", CB["green"]),
         (esZ, r"breaking  $-\lambda\sum Z_j$", CB["red"])]):
    ax = fig.add_subplot(gs[k+1])
    for i in range(es.shape[1]):
        ax.plot(lam_es, es[:, i], lw=2, color=col, alpha=0.85)
    ax.set_xlabel(r"$\lambda$"); ax.set_ylabel(r"$-\ln\lambda_i$")
    ax.set_title(ttl, fontsize=11, color=col)
    ax.set_ylim(0, 3.4)
    spread = es[:, -1]-es[:, 0]
    ax.text(0.03, 0.93, f"spread at $\\lambda$=0.6:  {spread[-1]:.2f}",
            transform=ax.transAxes, fontsize=9.5, va="top", color=col)
fig.suptitle("Fig. 5   Entanglement spectrum of the half-ring", fontsize=12.5, y=1.03)
plt.tight_layout(); plt.show()

## Fig. 6 — the projective representation

On the open chain the symmetry generators, restricted to the ground space, factorise into a left piece and a right piece:

$$U_{\mathrm{odd}}\big|_{\mathrm{GS}}=\underbrace{(X_1Z_2)}_{L_{\mathrm{odd}}}\cdot\underbrace{(\cdots)}_{R_{\mathrm{odd}}},
\qquad
U_{\mathrm{even}}\big|_{\mathrm{GS}}=\underbrace{(Z_1)}_{L_{\mathrm{even}}}\cdot\underbrace{(\cdots)}_{R_{\mathrm{even}}} .$$

The two full operators **commute**, but each edge factor pair **anticommutes**. A $\mathbb{Z}_2\times\mathbb{Z}_2$ acting with an anticommuting pair has no one-dimensional representation, so each edge must carry at least two states. That is the protection, and $2\times2=4$ is the degeneracy of Fig. 2.

In [ ]:
N = 8
S = stabilizers(N)
P = np.eye(2**N)
for s in S:
    P = P @ ((np.eye(2**N) + s)/2)

Uo, Ue = U_sub(N,0), U_sub(N,1)
Lo, Ro = op(N,{0:X,1:Z}), op(N,{N-1:Z})
Le, Re = op(N,{0:Z}),     op(N,{N-2:Z,N-1:X})

checks = [
 (r"$[U_{\mathrm{odd}},U_{\mathrm{even}}]=0$", np.allclose(Uo@Ue, Ue@Uo)),
 (r"$U_{\mathrm{odd}}=L_{\mathrm{odd}}R_{\mathrm{odd}}$ on GS",  np.allclose(P@Uo@P, P@(Lo@Ro)@P)),
 (r"$U_{\mathrm{even}}=L_{\mathrm{even}}R_{\mathrm{even}}$ on GS", np.allclose(P@Ue@P, P@(Le@Re)@P)),
 (r"$\{L_{\mathrm{odd}},L_{\mathrm{even}}\}=0$  (left edge)",  np.allclose(Lo@Le, -Le@Lo)),
 (r"$\{R_{\mathrm{odd}},R_{\mathrm{even}}\}=0$  (right edge)", np.allclose(Ro@Re, -Re@Ro)),
]

fig, ax = plt.subplots(figsize=(8.6, 3.0)); ax.axis("off")
for i, (lab, ok) in enumerate(checks):
    y = 1 - i*0.2
    ax.text(0.02, y, lab, fontsize=11.5, va="center")
    ax.text(0.80, y, "TRUE" if ok else "FALSE", fontsize=11.5, va="center",
            color=CB["green"] if ok else CB["red"], fontweight="bold")
ax.text(0.02, 1.22, "Fig. 6   Global commutation, local anticommutation",
        fontsize=12.5, va="center")
ax.set_xlim(0,1); ax.set_ylim(-0.08, 1.32)
plt.tight_layout(); plt.show()

print("\nThe symmetry is an ordinary representation globally,")
print("but a projective one on each edge -- the two cocycles cancel.")

## Fig. 7 — decorated domain walls

Reading the stabilizer for odd $j$ as
$$X_j = Z_{j-1}Z_{j+1}$$
says: the odd spin at $j$ is flipped exactly when the two even neighbours disagree, i.e. **exactly on a domain wall of the even sublattice**. The cluster state is the domain walls of one $\mathbb{Z}_2$ decorated with charges of the other.

This is the construction that generalises: in the "Higgs $=$ SPT" setting the domain walls belong to a higher-form symmetry, but the logic is unchanged.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 2.6)); ax.axis("off")
cfg = [1,1,-1,-1,-1,1,1,-1]     # even-sublattice spins
n = len(cfg)
for i, s in enumerate(cfg):
    x = 2*i
    ax.add_patch(Circle((x, 0), 0.22, color=CB["blue"], zorder=3))
    ax.annotate("", xy=(x, 0.28*s+0.16*s), xytext=(x, -0.16*s),
                arrowprops=dict(arrowstyle="-|>", color="white", lw=2), zorder=4)
    if i < n-1 and cfg[i] != cfg[i+1]:
        ax.add_patch(Circle((x+1, 0), 0.20, color=CB["red"], zorder=3))
        ax.text(x+1, 0.45, r"$X=-1$", ha="center", fontsize=9, color=CB["red"])
        ax.plot([x+1, x+1], [-0.75, -0.32], color=CB["red"], lw=1.4, ls="--")
        ax.text(x+1, -0.95, "domain\nwall", ha="center", fontsize=8.5, color=CB["red"])
    elif i < n-1:
        ax.add_patch(Circle((x+1, 0), 0.20, color=CB["orange"], alpha=0.55, zorder=3))
        ax.text(x+1, 0.45, r"$X=+1$", ha="center", fontsize=9, color=CB["orange"])
ax.plot([0, 2*(n-1)], [0,0], color=CB["grey"], lw=1.1, zorder=1)
ax.set_xlim(-1, 2*(n-1)+1); ax.set_ylim(-1.35, 0.95)
ax.set_title("Fig. 7   Odd-site charges sit exactly on the domain walls of the even sublattice",
             fontsize=12, loc="left")
plt.tight_layout(); plt.show()

## Summary

| observation | figure | meaning |
|---|---|---|
| ground space is 4-fold on an open chain, for both parities of $N$ | 2, 3 | two protected states per edge |
| ground state is unique on a ring | 2 | the degeneracy lives on the boundary, not in the bulk |
| splitting is $\sim10^{-6}$ under $-\lambda\sum X$, $\sim10^{-1}$ under $-\lambda\sum Z$ | 4 | the symmetry, not the gap, is what protects |
| entanglement spectrum is exactly flat and 4-fold at $\lambda=0$ | 5 | the same projective structure, seen from the cut |
| the generators commute globally but anticommute on each edge | 6 | $H^2(\mathbb{Z}_2\times\mathbb{Z}_2,U(1))=\mathbb{Z}_2$ is nontrivial |
| odd-site charges sit on even-sublattice domain walls | 7 | decorated domain wall construction |

### Things to try next

* replace $-\lambda\sum Z_j$ by $-\lambda\sum Z_jZ_{j+1}$: it also breaks the symmetry, and the quartet splits again;
* push the symmetric field to $\lambda>1$: the bulk gap closes and reopens, and the system leaves the SPT phase — the degeneracy is lost through a **bulk** transition rather than an edge one;
* gauge one of the two $\mathbb{Z}_2$'s and watch the SPT turn into a symmetry-breaking phase with two bulk ground states.